<a href="https://colab.research.google.com/github/soham-never-codes/Festiva-AI-Event-Planner/blob/main/Festiva_Week6_Deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount + install ──────────────────────────────────────────────
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/festiva'

print("⬇️ Installing Server Dependencies...")
# [THE FIX: Added langchain, faiss, sentence-transformers, and scikit-learn]
!pip install -q fastapi uvicorn nest-asyncio pyngrok google-generativeai langchain langchain-community sentence-transformers faiss-cpu scikit-learn joblib

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['NGROK_AUTH_TOKEN'] = userdata.get('NGROK_AUTH_TOKEN')

print("✅ Environment ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⬇️ Installing Server Dependencies...
✅ Environment ready!


In [ ]:
%%writefile festiva_api.py
import os, json, re, joblib, numpy as np
import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import google.generativeai as genai
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

PROJECT_DIR = "/content/drive/MyDrive/festiva"
app = FastAPI(title="Festiva Planner API")

app.add_middleware(
    CORSMiddleware, allow_origins=["*"],
    allow_methods=["*"], allow_headers=["*"]
)

# ── Load all components at startup ────────────────────────────────────
print("Loading ML Models into Server Memory...")

model    = joblib.load(f"{PROJECT_DIR}/models/budget_model.pkl")
scaler   = joblib.load(f"{PROJECT_DIR}/models/scaler.pkl")
encoders = joblib.load(f"{PROJECT_DIR}/models/encoders.pkl")
meta     = joblib.load(f"{PROJECT_DIR}/models/metadata.pkl")

SPEND_COLS = meta["spend_cols"]
FEATURE_COLS = meta["feature_cols"]

# Hardware check
device_str = "cuda" if torch.cuda.is_available() else "cpu"
device_id = 0 if torch.cuda.is_available() else -1

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device_str}
)
vectorstore = FAISS.load_local(
    f"{PROJECT_DIR}/rag/faiss_index", embedder,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

nlp_clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device_id)

# Initialize Gemini
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
llm = genai.GenerativeModel('gemini-2.5-flash', generation_config={"response_mime_type": "application/json"})

CITIES = ["bangalore","bengaluru","mumbai","delhi","chennai","hyderabad","pune"]
CITY_NORM = {"bengaluru":"Bangalore"}
LABELS = ["wedding","corporate event","birthday party","engagement","conference"]

def parse_input(text):
    entities = {}
    for pat, mult in [(r"₹\s*(\d+(?:\.\d+)?)\s*(?:L\b|lakh|lakhs)", 1e5),
                      (r"(\d+(?:\.\d+)?)\s*(?:L\b|lakh|lakhs)", 1e5),
                      (r"₹\s*([\d,]+)", 1)]:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            entities["budget"] = int(float(m.group(1).replace(",","")) * mult)
            break

    for city in CITIES:
        if city in text.lower():
            entities["city"] = CITY_NORM.get(city, city.capitalize())
            break

    m = re.search(r"(\d+)\s+(?:guests?|people|attendees?)", text, re.IGNORECASE)
    if m:
        entities["guest_count"] = int(m.group(1))

    res = nlp_clf(text, LABELS)
    etype = res["labels"][0].replace(" party","").replace(" event","").replace(" ","_")

    return {"event_type":etype, "city":entities.get("city","Bangalore"),
            "budget":entities.get("budget",500000), "guests":entities.get("guest_count",100)}

class PlanRequest(BaseModel):
    query: str

class AskRequest(BaseModel):
    question: str

@app.get("/")
def root():
    return {"message": "Festiva API running", "status": "ok"}

@app.get("/health")
def health():
    return {"status":"ok"}

@app.post("/plan")
def plan(req: PlanRequest):
    parsed = parse_input(req.query)

    # 1. Budget ML
    try:
        row = np.array([[encoders["event_type"].transform([parsed["event_type"]])[0],
                         encoders["city"].transform([parsed["city"]])[0],
                         parsed["guests"], parsed["budget"], 1,
                         encoders["season"].transform(["winter"])[0]]])
    except:
        row = np.array([[0,0,parsed["guests"],parsed["budget"],1,0]])

    pcts = model.predict(scaler.transform(row))[0]
    pcts = np.clip(pcts,0,None); pcts /= pcts.sum()
    budget_breakdown = {c:{"pct":round(p*100,1),"amount":int(p*parsed["budget"])}
                        for c,p in zip(SPEND_COLS,pcts)}

    # 2. RAG
    docs = retriever.invoke(f"{parsed['event_type']} planning {parsed['city']}")
    context = "\n\n".join(d.page_content for d in docs)

    budget_str = "\n".join(f"{k}: ₹{v['amount']:,} ({v['pct']}%)" for k, v in budget_breakdown.items())

    # 3. LLM Plan (Gemini)
    prompt = f"""
    You are Festiva, an Indian event planner. Return JSON only with exactly these keys:
    "summary" (string), "timeline" (list of objects with 'week', 'tasks'),
    "checklist" (list of strings), "vendor_categories" (list of objects with 'category', 'priority', 'tip'), "tips" (list of strings).
    Plan: {req.query}
    Detected: {parsed}
    Budget: {budget_str}
    Knowledge: {context[:1200]}
    """

    resp = llm.generate_content(prompt)
    plan_data = json.loads(resp.text)

    return {"detected":parsed,"budget_breakdown":budget_breakdown, "plan":plan_data}

@app.post("/ask")
def ask(req: AskRequest):
    docs = retriever.invoke(req.question)
    return {"question":req.question,"answer":"\n\n".join(d.page_content for d in docs)}

Overwriting festiva_api.py


In [ ]:
#  Launch with ngrok public URL ─────────────────────────────────
import nest_asyncio, uvicorn, threading
from pyngrok import ngrok, conf
nest_asyncio.apply()

# Configure ngrok securely
conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']

# Close any existing tunnels to prevent errors
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("\n" + "🚀" * 25)
print(f"Festiva API is LIVE at: {public_url}")
print(f"Test UI / Swagger:      {public_url}/docs")
print("🚀" * 25 + "\n")

# Start server in background thread so Colab doesn't freeze
def run():
    uvicorn.run("festiva_api:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
Festiva API is LIVE at: https://scorpion-tweed-protrude.ngrok-free.dev
Test UI / Swagger:      https://scorpion-tweed-protrude.ngrok-free.dev/docs
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀



INFO:     Started server process [4553]


In [ ]:
#  Test the live API ─────────────────────────────────────────────
# ── CELL 4: Test the live API (With Smart Polling) ────────────────────────
import requests, json, time

BASE = str(public_url)

print("⏳ Waiting for the AI models to load into server memory...")
print("   (This usually takes about 15-20 seconds)")

# Smart Polling: Try to hit the /health endpoint every 2 seconds
server_ready = False
for i in range(15):
    try:
        r_health = requests.get(f"{BASE}/health")
        if r_health.status_code == 200:
            print("✅ Server is awake and healthy!\n")
            server_ready = True
            break
    except Exception:
        pass # Ignore the error and try again
    time.sleep(2)

if not server_ready:
    print("❌ Error: The server never woke up. It likely crashed in the background.")
else:
    # Test 2: The Orchestrator
    payload = {"query": "Wedding in Bangalore, budget ₹10L, 200 guests"}
    print(f"📡 Sending POST request to {BASE}/plan...")

    response = requests.post(f"{BASE}/plan", json=payload)

    if response.status_code == 200:
        result = response.json()
        print("\n🎉 FESTIVA LIVE API RESULT")
        print("=" * 50)
        print(f"Event: {result['detected']['event_type']} in {result['detected']['city']}")
        print(f"Budget: ₹{result['detected']['budget']:,}")
        print(f"\nSummary: {result['plan'].get('summary','')}")
        print("\nBudget breakdown:")
        for cat, v in list(result['budget_breakdown'].items())[:5]:
            print(f"  {cat}: ₹{v['amount']:,} ({v['pct']}%)")
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text)

⏳ Waiting for the AI models to load into server memory...
   (This usually takes about 15-20 seconds)
INFO:     34.169.49.80:0 - "GET /health HTTP/1.1" 200 OK
✅ Server is awake and healthy!

📡 Sending POST request to https://scorpion-tweed-protrude.ngrok-free.dev/plan...
INFO:     2409:40f2:34a:4eef:2030:9f21:9971:14c8:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2409:40f2:34a:4eef:2030:9f21:9971:14c8:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     34.169.49.80:0 - "POST /plan HTTP/1.1" 200 OK

🎉 FESTIVA LIVE API RESULT
Event: wedding in Bangalore
Budget: ₹1,000,000

Summary: Festiva is thrilled to help you plan your exquisite Indian wedding in Bangalore! With a budget of ₹10 Lakhs and 200 guests, a detailed planning timeline of 6-12 months is highly recommended to ensure every detail is perfect. This plan focuses on optimal budget allocation for key categories like venue, catering, and decor, ensuring a memorable celebration.

Budget breakdown:
  venue: ₹303,866 (30.4%)
  catering: ₹25

In [ ]:
%%writefile app.py
import streamlit as st
import requests
import time

# ── PAGE CONFIG ──────────────────────────────────────────────────────────
st.set_page_config(page_title="Festiva | AI Event Planner", page_icon="🎊", layout="wide")

# ── SIDEBAR (Configuration) ──────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ System Config")
    st.markdown("Connect the frontend to your FastAPI backend.")
    # The user can paste their active ngrok URL here
    api_url = st.text_input("Backend API URL", placeholder="https://....ngrok-free.app")

    st.markdown("---")
    st.markdown("**Powered by:**\n- Gemini 2.5 Flash\n- XGBoost\n- FAISS Vector DB")

# ── MAIN UI ──────────────────────────────────────────────────────────────
st.title("🎊 Festiva: Your AI Event Architect")
st.markdown("Describe your dream event, and our Multi-Agent AI will handle the budget, timeline, and vendors.")

# User Input Form
with st.form("planning_form"):
    query = st.text_area("What are you planning?", placeholder="e.g., I want to plan a corporate conference in Bangalore for 300 people with a budget of ₹20 Lakhs.", height=100)
    submitted = st.form_submit_button("Generate Master Plan 🚀")

# ── LOGIC & DISPLAY ──────────────────────────────────────────────────────
if submitted:
    if not api_url:
        st.error("⚠️ Please enter your Backend API URL in the sidebar first!")
    elif not query:
        st.warning("⚠️ Please describe your event!")
    else:
        # Show a cool loading spinner while the backend does the math
        with st.spinner("AI Agents are calculating budgets and reading knowledge bases..."):
            try:
                # Send the request to your FastAPI backend
                response = requests.post(f"{api_url.rstrip('/')}/plan", json={"query": query})

                if response.status_code == 200:
                    data = response.json()
                    plan = data["plan"]
                    detected = data["detected"]
                    budget = data["budget_breakdown"]

                    st.success("✅ Plan Generated Successfully!")

                    # --- DISPLAY RESULTS IN BEAUTIFUL COLUMNS ---

                    # 1. High-Level Summary
                    st.subheader("📝 Executive Summary")
                    st.write(plan.get("summary", "No summary provided."))

                    # 2. Key Metrics & Budget
                    st.markdown("---")
                    st.subheader("💰 AI Budget Optimization")

                    col1, col2, col3 = st.columns(3)
                    col1.metric("Event Type", detected['event_type'].replace('_', ' ').title())
                    col2.metric("City", detected['city'])
                    col3.metric("Total Budget", f"₹{detected['budget']:,}")

                    # Render progress bars for the budget
                    for category, details in budget.items():
                        if details['pct'] > 0:
                            st.markdown(f"**{category.title()}**: ₹{details['amount']:,} ({details['pct']}%)")
                            st.progress(details['pct'] / 100.0)

                    # 3. Timeline & Checklist Side-by-Side
                    st.markdown("---")
                    colA, colB = st.columns(2)

                    with colA:
                        st.subheader("📅 Action Timeline")
                        for item in plan.get("timeline", []):
                            with st.expander(item.get("week", "Task Phase")):
                                for task in item.get("tasks", []):
                                    st.markdown(f"- {task}")

                    with colB:
                        st.subheader("✅ Crucial Checklist")
                        for item in plan.get("checklist", []):
                            st.checkbox(item) # Makes interactive checkboxes!

                    # 4. Vendor Advice
                    st.markdown("---")
                    st.subheader("🤝 Vendor Negotiation Strategies")
                    for vendor in plan.get("vendor_categories", []):
                        st.info(f"**[{vendor.get('priority', 'Normal')}] {vendor.get('category', 'Vendor')}**\n\n{vendor.get('tip', '')}")

                else:
                    st.error(f"Backend Error {response.status_code}: {response.text}")

            except Exception as e:
                st.error(f"Failed to connect to the backend. Make sure your ngrok URL is correct. Error: {e}")

Writing app.py


In [ ]:
import subprocess
import time
from google.colab import output

# 1. Forcefully kill the stuck server
!fuser -k 8501/tcp

# 2. Start Streamlit with CORS and XSRF disabled so Colab's iframe can connect
print("⚙️ Booting up Streamlit server...")
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# 3. Give the WebSockets 3-5 seconds to fully initialize
time.sleep(4)

# 4. Embed the app
print("🌐 Embedding Festiva UI below...")
output.serve_kernel_port_as_window(8501)

8501/tcp:            13766
⚙️ Booting up Streamlit server...
🌐 Embedding Festiva UI below...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>